# ECG Self-Supervised Encoder -- PTB-XL

Refactored to match the engineering standard of `image-encoder.ipynb`
(the reference implementation). The ECG-specific research choices --
architecture, augmentations, SimCLR / MultiSupCon objectives, dataset,
preprocessing -- are unchanged. Everything else (TPU training loop,
checkpointing, early stopping, evaluation, NT-Xent efficiency, scheduler,
MLflow, naming) has been brought up to the same standard as the image
notebook.




In [ ]:
# ==========================================
# Environment check (Kaggle TPU VM v5e-8)
# ==========================================
# Kaggle's TPU VM base image ships torch + torch_xla preinstalled and
# version-matched to the host libtpu, and wfdb/dagshub/mlflow are not
# preinstalled -- install just those instead of touching torch/torch_xla.
!pip install -q wfdb dagshub mlflow

import torch
import torch_xla

print("torch      :", torch.__version__)
print("torch_xla  :", torch_xla.__version__)




In [ ]:
# ==========================================
# Basic Imports
# ==========================================

import os
import ast as ast_module
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, filtfilt
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ==========================================
# PyTorch
# ==========================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==========================================
# TPU Support
# ==========================================

import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.runtime as xr

# ==========================================
# Evaluation
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# ==========================================
# Experiment tracking
# ==========================================

import dagshub
import mlflow

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100




In [ ]:
# ==========================================
# Reproducibility
# ==========================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

# ==========================================
# TPU Device
# ==========================================

# Improves matmul throughput on TPU (uses bf16 internally for fp32 matmuls)
torch.set_float32_matmul_precision("high")

DEVICE = xm.xla_device()

print("Device:", DEVICE)
print("Supported devices:", xm.get_xla_supported_devices())

# NOTE: a Kaggle TPU v5e-8 exposes 8 independent chips, but a plain
# xm.xla_device() call (same as used here) only claims a single chip.
# That is enough for this notebook and keeps the code linear and directly
# comparable to the image notebook. To use all 8 chips, wrap the notebook
# body in a function and launch it with
# torch_xla.distributed.xla_multiprocessing.spawn(fn, nprocs=8), scaling
# batch_size / lr accordingly.




In [ ]:
# ==========================================
# Project Config
# ==========================================
# All previously-hardcoded / scattered constants are consolidated here so
# the notebook has a single source of truth for hyperparameters, mirroring
# CFG in the image notebook.

CFG = {
    # ---- data ----
    "sampling_rate": 100,           # Hz
    "signal_length": 1000,          # timesteps (10s @ 100Hz)
    "n_leads": 12,
    "superclasses": ["NORM", "MI", "STTC", "CD", "HYP"],
    "meta_features": ["age", "sex", "weight", "nurse", "site", "device"],

    # ---- encoder / projection dims (UNCHANGED -- architecture is frozen) ----
    "embedding_dim": 256,           # ECGEncoder.feature_dim
    "projection_dim": 128,          # SimCLR / SupCon projector output

    # ---- SimCLR pretraining (UNCHANGED objective/temperature) ----
    "simclr_batch_size": 512,
    "simclr_lr": 1e-3,
    "simclr_weight_decay": 1e-4,
    "simclr_temperature": 0.2,      # unchanged -- do not touch the objective
    "simclr_epochs": 100,
    "simclr_warmup_epochs": 5,

    # ---- MultiSupCon pretraining (UNCHANGED objective/temperature) ----
    "supcon_batch_size": 1024,
    "supcon_lr": 3e-4,
    "supcon_weight_decay": 1e-4,
    "supcon_temperature": 0.2,      # unchanged -- do not touch the objective
    "supcon_epochs": 100,
    "supcon_warmup_epochs": 5,

    # ---- linear classifier heads (frozen encoder + trainable head) ----
    "clf_batch_size": 512,
    "clf_lr": 1e-3,
    "clf_epochs": 20,
    "clf_patience": 3,

    # ---- eval / checkpoint cadence (shared across pretraining phases) ----
    "checkpoint_every": 5,          # save a resumable checkpoint every N epochs
    "eval_every": 5,                # run the linear probe every N epochs
    "pretrain_patience": 3,         # early-stop patience (in evals) on probe AUROC

    # ---- dataloader ----
    "num_workers": 4,
}

CFG




In [ ]:
# ==========================================
# Dataset Paths
# ==========================================

DATA_ROOT = Path("/kaggle/input/datasets/garethwmch")
DATA_DIR = DATA_ROOT / "ptb-xl-1-0-3" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

print(DATA_DIR)
print(os.listdir(DATA_ROOT))




In [ ]:
# ==========================================
# MLflow / Dagshub experiment tracking setup
# ==========================================
# Initialized once, up front, so every training phase below can open an
# mlflow.start_run() block without re-initializing the tracking backend.

dagshub.init(repo_owner="RanbeerReddy", repo_name="Multimodal-classifier", mlflow=True)




In [ ]:
# ==========================================
# Load raw ECG signals + annotation metadata
# ==========================================

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(str(path) + "/" + f) for f in tqdm(df.filename_lr)]
    else:
        data = [wfdb.rdsamp(str(path) + "/" + f) for f in tqdm(df.filename_hr)]
    data = np.array([signal for signal, meta in data])
    return data

Y = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
Y.scp_codes = Y.scp_codes.apply(lambda x: ast_module.literal_eval(x))

X = load_raw_data(Y, CFG["sampling_rate"], DATA_DIR)

print(f"ECG signals shape : {X.shape}")   # (21837, 1000, 12)
print(f"Metadata shape    : {Y.shape}")
print(f"Sampling rate     : {CFG['sampling_rate']} Hz -> {X.shape[1] / CFG['sampling_rate']}s per record")




In [ ]:
# ==========================================
# Diagnostic superclass labels + filtering
# ==========================================
# (previously duplicated across two cells that computed the same mask --
# consolidated into one pass over X/Y)

SUPERCLASSES = CFG["superclasses"]

agg_df = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]
print(f"Diagnostic SCP codes : {len(agg_df)}")

def aggregate_superclass(y_dic):
    return list(set(
        agg_df.loc[key].diagnostic_class
        for key in y_dic.keys()
        if key in agg_df.index
    ))

Y["diagnostic_superclass"] = Y.scp_codes.apply(aggregate_superclass)
Y["n_superclass"] = Y["diagnostic_superclass"].apply(len)

mask_labeled = Y["n_superclass"].values > 0
n_dropped = (~mask_labeled).sum()
print(f"Records with no superclass label: {n_dropped} ({100 * n_dropped / len(Y):.1f}%) -> excluded")

X = X[mask_labeled]
Y = Y[mask_labeled].copy()

for cls in SUPERCLASSES:
    Y[cls] = Y["diagnostic_superclass"].apply(lambda x: int(cls in x))

print(f"Signal shape after filtering: {X.shape}")
print("\nLabel distribution:")
for cls in SUPERCLASSES:
    n = Y[cls].sum()
    print(f"  {cls:4s}: {n:5d}  ({100 * n / len(Y):.1f}%)")




In [ ]:
# ==========================================
# Clean + impute metadata (English-only, per-column strategy documented)
# ==========================================

META_FEATURES = CFG["meta_features"]

COLS_TO_DROP = [
    "height",                # 68% missing
    "heart_axis",             # 38.9% missing, categorical
    "infarction_stadium1", "infarction_stadium2",
    "baseline_drift", "static_noise", "burst_noise",
    "electrodes_problems", "extra_beats", "pacemaker",
    "report",
    "recording_date",
    "validated_by", "second_opinion", "initial_autogenerated_report", "validated_by_human",
    "filename_lr", "filename_hr",
    "scp_codes", "diagnostic_superclass", "n_superclass",
    "patient_id",
]
COLS_TO_DROP = [c for c in COLS_TO_DROP if c in Y.columns]
Y_clean = Y.drop(columns=COLS_TO_DROP).copy()

# --- Imputation ---
age_median = Y_clean["age"].median()
Y_clean["age"] = Y_clean["age"].fillna(age_median)
print(f"age    -> median imputed: {age_median:.1f} years")

weight_median = Y_clean["weight"].median()
Y_clean["weight"] = Y_clean["weight"].fillna(weight_median)
print(f"weight -> median imputed: {weight_median:.1f} kg")

nurse_mode = Y_clean["nurse"].mode()[0]
Y_clean["nurse"] = Y_clean["nurse"].fillna(nurse_mode)
print(f"nurse  -> mode imputed: {nurse_mode}")

site_mode = Y_clean["site"].mode()[0]
Y_clean["site"] = Y_clean["site"].fillna(site_mode)
print(f"site   -> mode imputed: {site_mode}")

Y_clean["device"] = Y_clean["device"].astype("category").cat.codes
print(f"device -> label encoded ({Y['device'].nunique()} devices -> int)")

remaining_na = Y_clean[META_FEATURES].isnull().sum()
print(f"\nRemaining NaNs in META_FEATURES:\n{remaining_na}")




In [ ]:
# ==========================================
# Preprocessing: bandpass filter + per-sample z-score normalization
# ==========================================

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100, order=4):
    """Butterworth bandpass filter applied on all 12 leads.
    signal: (1000, 12) -> Returns: (1000, 12) filtered"""
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal, axis=0)


def normalize_per_sample(signal):
    """Z-score normalization per sample per lead. signal: (1000, 12)"""
    mean = signal.mean(axis=0, keepdims=True)
    std = signal.std(axis=0, keepdims=True) + 1e-8
    return (signal - mean) / std


def preprocess_ecg_batch(X_batch, fs=100, apply_filter=True):
    """Apply the full preprocessing pipeline on a batch of ECG signals.
    X_batch: (N, 1000, 12) -> Returns: (N, 1000, 12) preprocessed"""
    X_out = np.empty_like(X_batch, dtype=np.float32)
    for i in range(len(X_batch)):
        sig = X_batch[i].copy().astype(np.float64)
        if apply_filter:
            sig = bandpass_filter(sig, fs=fs)
        sig = normalize_per_sample(sig)
        X_out[i] = sig.astype(np.float32)
    return X_out

print("Preprocessing functions defined")
print("  - Bandpass filter : 0.5 Hz - 40 Hz  (Butterworth order 4)")
print("  - Normalization   : z-score per lead per sample")




In [ ]:
# ==========================================
# Visual demonstration: before vs after preprocessing
# ==========================================

sample_idx = 1
raw_signal = X[sample_idx]
proc_signal = preprocess_ecg_batch(X[sample_idx:sample_idx + 1])[0]

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
time = np.arange(CFG["signal_length"]) / CFG["sampling_rate"]
leads_to_show = [0, 1, 6]  # Lead I, II, V1

fig, axes = plt.subplots(len(leads_to_show), 2, figsize=(16, 7))
for row, lead_i in enumerate(leads_to_show):
    axes[row, 0].plot(time, raw_signal[:, lead_i], color="#1565C0", linewidth=0.8)
    axes[row, 0].set_ylabel(LEAD_NAMES[lead_i], fontweight="bold")
    axes[row, 0].set_title("Raw (mV)" if row == 0 else "", fontweight="bold")
    axes[row, 0].grid(True, alpha=0.3, color="#EF9A9A")

    axes[row, 1].plot(time, proc_signal[:, lead_i], color="#2E7D32", linewidth=0.8)
    axes[row, 1].set_ylabel(LEAD_NAMES[lead_i], fontweight="bold")
    axes[row, 1].set_title("Filtered + Normalized" if row == 0 else "", fontweight="bold")
    axes[row, 1].grid(True, alpha=0.3, color="#EF9A9A")

for ax in axes.flatten():
    ax.set_xlabel("Time (s)")
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("ECG Signal -- Before vs After Preprocessing", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Raw  -> mean={raw_signal.mean():.4f}, std={raw_signal.std():.4f}")
print(f"Proc -> mean={proc_signal.mean():.4f}, std={proc_signal.std():.4f}")




In [ ]:
# ==========================================
# Train / valid / test split (patient-aware via strat_fold)
# ==========================================

def get_split(Y_df, X_arr, folds):
    mask = np.isin(Y_df.strat_fold.values, folds)
    return X_arr[mask], Y_df[mask].copy()

x_train_raw, y_train = get_split(Y_clean, X, list(range(1, 9)))
x_valid_raw, y_valid = get_split(Y_clean, X, [9])
x_test_raw, y_test = get_split(Y_clean, X, [10])

print(f"Train : {len(y_train):5d} records  ({100 * len(y_train) / len(Y_clean):.1f}%)")
print(f"Valid : {len(y_valid):5d} records  ({100 * len(y_valid) / len(Y_clean):.1f}%)")
print(f"Test  : {len(y_test):5d} records  ({100 * len(y_test) / len(Y_clean):.1f}%)")




In [ ]:
# ==========================================
# Preprocess (with reproducible on-disk cache)
# ==========================================
# Previously the np.save() calls that build the cache were commented out
# while the *next* cell unconditionally np.load()'d from it -- that broke
# top-to-bottom reproducibility on a clean environment. Cache is now
# written and read consistently: if the cache exists, load it; otherwise
# compute it and save it for next time.

CACHE_FILES = {
    "train": Path("x_train.npy"),
    "valid": Path("x_valid.npy"),
    "test": Path("x_test.npy"),
}

if all(p.exists() for p in CACHE_FILES.values()):
    x_train = np.load(CACHE_FILES["train"])
    x_valid = np.load(CACHE_FILES["valid"])
    x_test = np.load(CACHE_FILES["test"])
    print("Loaded cached preprocessed arrays.")
else:
    print("Preprocessing train split...")
    x_train = preprocess_ecg_batch(x_train_raw)
    print("Preprocessing valid split...")
    x_valid = preprocess_ecg_batch(x_valid_raw)
    print("Preprocessing test split...")
    x_test = preprocess_ecg_batch(x_test_raw)

    np.save(CACHE_FILES["train"], x_train)
    np.save(CACHE_FILES["valid"], x_valid)
    np.save(CACHE_FILES["test"], x_test)
    print("Cached preprocessed arrays to disk.")

print(f"\nx_train : {x_train.shape}  dtype={x_train.dtype}")
print(f"x_valid : {x_valid.shape}  dtype={x_valid.dtype}")
print(f"x_test  : {x_test.shape}   dtype={x_test.dtype}")




In [ ]:
# ==========================================
# Normalize numeric metadata (fit on train, apply on all splits)
# ==========================================

NUM_META = ["age", "weight"]
meta_mean = y_train[NUM_META].mean()
meta_std = y_train[NUM_META].std() + 1e-8
for df in [y_train, y_valid, y_test]:
    df[NUM_META] = (df[NUM_META] - meta_mean) / meta_std

print(f"Train: {x_train.shape} | Valid: {x_valid.shape} | Test: {x_test.shape}")
print(f"Meta features: {META_FEATURES}")




In [ ]:
# ==========================================
# ECGDataset -- signal + demographics + labels (used by both classifiers)
# ==========================================

class ECGDataset(Dataset):
    """Multimodal dataset: ECG signal (12 leads) + demographics."""

    def __init__(self, X_signals, Y_meta_df, superclasses=SUPERCLASSES, meta_features=META_FEATURES):
        self.signals = X_signals.transpose(0, 2, 1).astype(np.float32)
        self.meta = Y_meta_df[meta_features].values.astype(np.float32)
        self.labels = Y_meta_df[superclasses].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        signal = torch.from_numpy(self.signals[idx])
        meta = torch.from_numpy(self.meta[idx])
        label = torch.from_numpy(self.labels[idx])
        return signal, meta, label


train_ds = ECGDataset(x_train, y_train)
valid_ds = ECGDataset(x_valid, y_valid)
test_ds = ECGDataset(x_test, y_test)

# Training: shuffle + drop_last is fine/standard (keeps batch shape fixed
# for XLA). Validation/test: NO shuffle, NO drop_last -- these must stay
# a fixed, reproducible, full-size evaluation set across runs.
clf_train_DataLoader = DataLoader(
    train_ds, batch_size=CFG["clf_batch_size"], shuffle=True,
    num_workers=CFG["num_workers"], persistent_workers=True, drop_last=True,
)
clf_valid_DataLoader = DataLoader(
    valid_ds, batch_size=CFG["clf_batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], persistent_workers=True, drop_last=False,
)
clf_test_DataLoader = DataLoader(
    test_ds, batch_size=CFG["clf_batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], persistent_workers=True, drop_last=False,
)

clf_train_loader = pl.MpDeviceLoader(clf_train_DataLoader, DEVICE)
clf_valid_loader = pl.MpDeviceLoader(clf_valid_DataLoader, DEVICE)
clf_test_loader = pl.MpDeviceLoader(clf_test_DataLoader, DEVICE)

# Sanity check on TPU
sig, meta, lab = next(iter(clf_train_loader))
print(f"Signal batch : {sig.shape}   -> (batch, 12 leads, 1000 timesteps)")
print(f"Meta batch   : {meta.shape}  -> (batch, {len(META_FEATURES)} features)")
print(f"Label batch  : {lab.shape}   -> (batch, {len(SUPERCLASSES)} classes)")




In [ ]:
# ==========================================
# SimCLR augmentation dataset (UNCHANGED augmentations)
# ==========================================

class ECGSSLData(Dataset):
    def __init__(self, X):
        self.X = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def augment(self, x):
        x = x.clone()

        # 1. Slight noise (always)
        x = x + 0.005 * torch.randn_like(x)

        # 2. Time shift (strong but consistent)
        shift = torch.randint(-50, 50, (1,))
        x = torch.roll(x, shifts=int(shift), dims=1)

        # 3. Scaling
        scale = torch.rand(1) * 0.4 + 0.8
        x = x * scale

        # 4. ONE strong transformation (not many)
        if torch.rand(1) < 0.5:
            # masking
            length = torch.randint(50, 150, (1,))
            start = torch.randint(0, 1000 - length, (1,))
            x[:, start:start + length] = 0
        else:
            # lead dropout
            lead = torch.randint(0, 12, (1,))
            x[lead] = 0

        return x

    def __getitem__(self, idx):
        x = self.X[idx]
        x1 = self.augment(x.clone())
        x2 = self.augment(x.clone())
        return x1, x2




In [ ]:
# ==========================================
# Depthwise-separable Conv1d + Residual Block (UNCHANGED)
# ==========================================

class DepthwiseSepConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, padding="same"):
        super().__init__()
        self.dw = nn.Conv1d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch, bias=False)
        self.pw = nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.GroupNorm(8, out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))))


class ResBlock1d(nn.Module):
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        self.conv1 = DepthwiseSepConv1d(channels, channels, kernel_size)
        self.conv2 = DepthwiseSepConv1d(channels, channels, kernel_size)
        self.bn = nn.GroupNorm(8, channels)

    def forward(self, x):
        return F.relu(self.bn(x + self.conv2(self.conv1(x))))




In [ ]:
# ==========================================
# ECG Encoder (UNCHANGED architecture -- 1D-CNN, GroupNorm, 256-d output)
# ==========================================

class ECGEncoder(nn.Module):
    def __init__(self, n_leads=CFG["n_leads"], base_channels=64, emb_dim=CFG["embedding_dim"]):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, base_channels, kernel_size=15, padding=7, bias=False),
            nn.GroupNorm(8, base_channels),
            nn.ReLU(),
            nn.MaxPool1d(2),
        )

        self.block1 = nn.Sequential(ResBlock1d(base_channels, kernel_size=7), nn.MaxPool1d(2))

        self.trans1 = nn.Sequential(
            nn.Conv1d(base_channels, base_channels * 2, 1, bias=False),
            nn.GroupNorm(8, base_channels * 2),
            nn.ReLU(),
        )

        self.block2 = nn.Sequential(ResBlock1d(base_channels * 2, kernel_size=5), nn.MaxPool1d(2))

        self.trans2 = nn.Sequential(
            nn.Conv1d(base_channels * 2, base_channels * 4, 1, bias=False),
            nn.GroupNorm(8, base_channels * 4),
            nn.ReLU(),
        )

        self.block3 = nn.Sequential(ResBlock1d(base_channels * 4, kernel_size=3), nn.MaxPool1d(2))

        self.gap = nn.AdaptiveAvgPool1d(1)

        # NOTE: renamed from `out_dim` -- `feature_dim` is the shared
        # attribute name both encoders (ECG / image) now expose for "raw
        # backbone output dimensionality" (see ResNet50Encoder.feature_dim
        # in the image notebook).
        self.feature_dim = base_channels * 4  # 256

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x: (B, 12, 1000)
        x = self.stem(x)
        x = self.block1(x)
        x = self.trans1(x)
        x = self.block2(x)
        x = self.trans2(x)
        x = self.block3(x)
        x = self.gap(x).squeeze(-1)  # (B, 256)
        return x




In [ ]:
# ==========================================
# Projection Head + SimCLR wrapper (UNCHANGED architecture)
# ==========================================

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, proj_dim=CFG["projection_dim"]):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        return self.net(x)


class ECGSimCLR(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.projector = ProjectionHead(encoder.feature_dim, CFG["projection_dim"])

    def forward(self, x):
        h = self.encoder(x)             # (B, 256)
        z = self.projector(h)           # (B, 128)
        z = F.normalize(z, dim=1)       # required for contrastive learning
        return h, z




In [ ]:
# ==========================================
# Classifier head (frozen encoder + demographic fusion) -- UNCHANGED
# ==========================================

class ECGClassifier(nn.Module):
    def __init__(self, encoder, n_meta=len(META_FEATURES), n_classes=len(SUPERCLASSES), dropout=0.3):
        super().__init__()
        self.encoder = encoder  # frozen

        self.meta_mlp = nn.Sequential(
            nn.Linear(n_meta, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        fusion_dim = encoder.feature_dim + 32

        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, signal, meta):
        with torch.no_grad():  # freeze encoder
            x = self.encoder(signal)
        m = self.meta_mlp(meta)
        x = torch.cat([x, m], dim=1)
        return self.head(x)




In [ ]:
# ==========================================
# NT-Xent Loss (SimCLR) -- cached masks/labels, same math as before
# ==========================================

class NTXentLoss(nn.Module):
    def __init__(self, temperature=CFG["simclr_temperature"]):
        super().__init__()
        self.temperature = temperature
        self._cached_batch_size = None
        self._mask = None
        self._labels = None

    def _build_cache(self, batch_size, device):
        self._mask = torch.eye(2 * batch_size, dtype=torch.bool, device=device)
        # same positive-pair construction as before: view-1 index i's
        # positive is view-2 index i (offset by B in the concatenated
        # batch), and vice versa.
        self._labels = torch.cat([
            torch.arange(batch_size, 2 * batch_size),
            torch.arange(batch_size),
        ]).to(device)
        self._cached_batch_size = batch_size

    def forward(self, z1, z2):
        # explicit fp32 cast -- don't rely on XLA autocast's op-allowlist
        # for normalize + matmul + cross_entropy
        z1 = F.normalize(z1.float(), dim=1)
        z2 = F.normalize(z2.float(), dim=1)

        batch_size = z1.size(0)
        z = torch.cat([z1, z2], dim=0)

        similarity = torch.matmul(z, z.T) / self.temperature

        if self._cached_batch_size != batch_size:
            self._build_cache(batch_size, similarity.device)

        similarity = similarity.masked_fill(self._mask, torch.finfo(similarity.dtype).min)
        return F.cross_entropy(similarity, self._labels)




In [ ]:
# ==========================================
# SimCLR dataset / dataloader / model / criterion
# ==========================================

ssl_dataset = ECGSSLData(x_train)

ssl_DataLoader = DataLoader(
    ssl_dataset,
    batch_size=CFG["simclr_batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    persistent_workers=True,
    drop_last=True,
)
ssl_loader = pl.MpDeviceLoader(ssl_DataLoader, DEVICE)

model = ECGSimCLR(ECGEncoder()).to(DEVICE)
criterion = NTXentLoss(temperature=CFG["simclr_temperature"])

# Sanity check on TPU
x1, x2 = next(iter(ssl_loader))
with torch.no_grad():
    _, z1 = model(x1)
    _, z2 = model(x2)
print("View batch  :", x1.shape)
print("Sanity loss :", criterion(z1, z2).item())




In [ ]:
# ==========================================
# Optimizer + Scheduler (warmup -> cosine decay)
# ==========================================
# The original SimCLR loop had no LR schedule at all (constant lr=1e-3).
# Ported the image notebook's warmup+cosine SequentialLR here per item 7
# of the refactor spec -- base lr and epoch count are unchanged, only the
# LR *trajectory* over training is added.

optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG["simclr_lr"], weight_decay=CFG["simclr_weight_decay"]
)

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, total_iters=CFG["simclr_warmup_epochs"]
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["simclr_epochs"] - CFG["simclr_warmup_epochs"]
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[CFG["simclr_warmup_epochs"]]
)

print(optimizer)
print(scheduler)




In [ ]:
# ==========================================
# Training step (TPU, bf16 autocast)
# ==========================================
# Ported from the image notebook's train_one_epoch: bf16 autocast around
# the forward pass, grad clipping, a single xm.optimizer_step(barrier=True)
# per step with NO follow-up xm.mark_step() (barrier=True already
# synchronizes -- the extra mark_step() in the original loop was a
# redundant second device sync every step).

def train_one_epoch(model, loader, optimizer, criterion, log_every=50):
    model.train()
    running_loss = torch.zeros((), device=DEVICE)
    num_batches = 0

    progress_bar = tqdm(loader)
    for x1, x2 in progress_bar:
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="xla", dtype=torch.bfloat16):
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss = criterion(z1, z2)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        xm.optimizer_step(optimizer, barrier=True)
        # (no separate xm.mark_step() -- barrier=True already does that)

        running_loss += loss.detach()
        num_batches += 1

        if num_batches % log_every == 0:
            progress_bar.set_postfix(loss=f"{(running_loss / num_batches).item():.4f}")

    return (running_loss / num_batches).item()




In [ ]:
# ==========================================
# Linear-probe AUROC harness -- the actual downstream-signal-based
# selection criterion (replaces "best = lowest SSL loss").
# ==========================================
# NT-Xent loss can keep falling while representation quality plateaus or
# degrades; checkpoint selection below is driven by this probe instead.

probe_train_df = y_train.sample(n=min(3000, len(y_train)), random_state=SEED)
probe_train_idx = y_train.index.get_indexer(probe_train_df.index)

probe_train_ds = ECGDataset(x_train[probe_train_idx], probe_train_df)
probe_valid_ds = ECGDataset(x_valid, y_valid)

probe_train_loader = pl.MpDeviceLoader(
    DataLoader(probe_train_ds, batch_size=256, shuffle=False, num_workers=2), DEVICE
)
probe_valid_loader = pl.MpDeviceLoader(
    DataLoader(probe_valid_ds, batch_size=256, shuffle=False, num_workers=2), DEVICE
)


@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    feats, labels = [], []
    for signal, meta, y in loader:
        f = encoder(signal)
        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()


def linear_probe_auroc(encoder, label_cols=SUPERCLASSES, verbose=True):
    """Freeze encoder, fit a linear head per label on frozen features,
    return mean AUROC on the held-out valid split."""
    X_train_feat, y_train_feat = extract_features(encoder, probe_train_loader)
    X_valid_feat, y_valid_feat = extract_features(encoder, probe_valid_loader)

    per_label = {}
    for i, col in enumerate(label_cols):
        y_tr, y_va = y_train_feat[:, i], y_valid_feat[:, i]
        if len(np.unique(y_tr)) < 2 or len(np.unique(y_va)) < 2:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
        clf.fit(X_train_feat, y_tr)
        preds = clf.predict_proba(X_valid_feat)[:, 1]
        per_label[col] = roc_auc_score(y_va, preds)

    if verbose:
        for col, score in per_label.items():
            xm.master_print(f"      {col}: {score:.4f}")
        skipped = [c for c in label_cols if c not in per_label]
        if skipped:
            xm.master_print(f"      (skipped, insufficient class variance: {skipped})")

    encoder.train()
    return float(np.mean(list(per_label.values()))) if per_label else float("nan")




In [ ]:
# ==========================================
# Checkpoint config + resume (defined BEFORE the training loop that
# references START_EPOCH -- avoids the NameError-on-fresh-run ordering
# bug found in the image notebook's own SimCLR section)
# ==========================================

CHECKPOINT_DIR = Path("checkpoints_simclr")
CHECKPOINT_DIR.mkdir(exist_ok=True)

CHECKPOINT_EVERY = CFG["checkpoint_every"]
EVAL_EVERY = CFG["eval_every"]
PATIENCE = CFG["pretrain_patience"]

START_EPOCH = 0
history = []
probe_history = []
best_auroc = -1.0
patience_counter = 0

resume_path = CHECKPOINT_DIR / "latest.pth"
if resume_path.exists():
    ckpt = torch.load(resume_path, map_location="cpu")

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])

    # Adam moments load as CPU tensors -- move them onto the XLA device or
    # you'll get a device-mismatch error on the next optimizer step.
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    scheduler.load_state_dict(ckpt["scheduler_state"])
    history = ckpt["history"]
    probe_history = ckpt.get("probe_history", [])
    best_auroc = ckpt.get("best_auroc", -1.0)
    patience_counter = ckpt.get("patience_counter", 0)
    START_EPOCH = ckpt["epoch"]

    xm.master_print(f"Resumed from checkpoint at epoch {START_EPOCH}")
else:
    xm.master_print("No checkpoint found -- starting fresh")




In [ ]:
# ==========================================
# SimCLR training loop (TPU) -- resume + checkpointing + AUROC-based
# early stopping, all wrapped in a single MLflow run
# ==========================================

with mlflow.start_run(run_name="simclr_pretrain"):
    mlflow.log_params({
        "encoder": "ECGEncoder",
        "ssl_method": "SimCLR",
        "batch_size": CFG["simclr_batch_size"],
        "lr": CFG["simclr_lr"],
        "weight_decay": CFG["simclr_weight_decay"],
        "temperature": CFG["simclr_temperature"],
        "epochs": CFG["simclr_epochs"],
        "warmup_epochs": CFG["simclr_warmup_epochs"],
        "embedding_dim": CFG["embedding_dim"],
        "projection_dim": CFG["projection_dim"],
        "eval_every": EVAL_EVERY,
        "patience": PATIENCE,
    })

    for epoch in range(START_EPOCH, CFG["simclr_epochs"]):
        train_loss = train_one_epoch(model, ssl_loader, optimizer, criterion)
        scheduler.step()
        history.append(train_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        xm.master_print(
            f"Epoch [{epoch + 1}/{CFG['simclr_epochs']}] "
            f"Loss: {train_loss:.4f} | LR: {current_lr:.2e}"
        )
        mlflow.log_metrics({"ssl_loss": train_loss, "lr": current_lr}, step=epoch)

        # ---- resumable checkpoint ----
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == CFG["simclr_epochs"]:
            xm.save({
                "epoch": epoch + 1,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "history": history,
                "probe_history": probe_history,
                "best_auroc": best_auroc,
                "patience_counter": patience_counter,
            }, CHECKPOINT_DIR / "latest.pth")
            xm.master_print(f"  -> checkpoint saved at epoch {epoch + 1}")

        # ---- linear-probe validation + early stopping on real signal ----
        if (epoch + 1) % EVAL_EVERY == 0 or (epoch + 1) == CFG["simclr_epochs"]:
            auroc = linear_probe_auroc(model.encoder)
            probe_history.append((epoch + 1, auroc))
            mlflow.log_metric("probe_auroc", auroc, step=epoch)
            xm.master_print(f"  -> linear probe mean AUROC: {auroc:.4f}")

            if auroc > best_auroc:
                best_auroc = auroc
                patience_counter = 0
                xm.save(model.encoder.state_dict(), CHECKPOINT_DIR / "encoder_best.pth")
                xm.master_print(f"  -> new best encoder saved (AUROC={auroc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    xm.master_print(
                        f"No AUROC improvement for {PATIENCE} evals -- stopping early at epoch {epoch + 1}"
                    )
                    break

    mlflow.log_metric("best_probe_auroc", best_auroc)
    mlflow.log_artifact(str(CHECKPOINT_DIR / "encoder_best.pth"))

# Debug-only snapshot of the final epoch's weights -- NOT used downstream.
# (Previously this unconditional save silently overwrote the "best"
# selection: cell 36 of the original notebook loaded THIS file instead of
# the best-AUROC checkpoint. Kept here only for debugging, clearly named.)
xm.save(model.encoder.state_dict(), "ecg_encoder_simclr_final.pth")




In [ ]:
# ==========================================
# Load frozen SimCLR encoder -- the BEST checkpoint, not the last epoch
# ==========================================

encoder = ECGEncoder()
encoder.load_state_dict(torch.load(CHECKPOINT_DIR / "encoder_best.pth", map_location="cpu"))
encoder = encoder.to(DEVICE)

for p in encoder.parameters():
    p.requires_grad = False
encoder.eval()

clf_model = ECGClassifier(encoder).to(DEVICE)

total_params = sum(p.numel() for p in clf_model.parameters())
trainable_params = sum(p.numel() for p in clf_model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")




In [ ]:
# ==========================================
# Class imbalance -> pos_weight for BCE, computed dynamically
# ==========================================
# Previously a hardcoded EDA snapshot (POS_WEIGHT = tensor([1.249, ...])),
# which silently goes stale if the split/filtering/labels change. Computed
# live from the current y_train distribution instead, same pattern as the
# image notebook's CheXpert pos_weight.

pos_counts = y_train[SUPERCLASSES].sum(axis=0).values
neg_counts = len(y_train) - pos_counts
POS_WEIGHT = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32)
print(dict(zip(SUPERCLASSES, POS_WEIGHT.tolist())))




In [ ]:
# ==========================================
# Classifier training step + evaluation (TPU, bf16) -- SHARED helpers,
# reused by both classifier phases (SimCLR-encoder and SupCon-encoder)
# ==========================================

def train_clf_epoch(model, loader, optimizer, criterion, log_every=50):
    model.train()
    running_loss = torch.zeros((), device=DEVICE)
    num_batches = 0

    progress_bar = tqdm(loader)
    for signal, meta, labels in progress_bar:
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="xla", dtype=torch.bfloat16):
            outputs = model(signal, meta)
            loss = criterion(outputs.float(), labels)

        loss.backward()
        xm.optimizer_step(optimizer, barrier=True)

        running_loss += loss.detach()
        num_batches += 1

        if num_batches % log_every == 0:
            progress_bar.set_postfix(loss=f"{(running_loss / num_batches).item():.4f}")

    return (running_loss / num_batches).item()


@torch.no_grad()
def evaluate_clf(model, loader, criterion, label_cols=SUPERCLASSES):
    model.eval()
    val_loss = torch.zeros((), device=DEVICE)
    num_batches = 0
    y_true, y_pred = [], []

    for signal, meta, labels in loader:
        outputs = model(signal, meta)
        loss = criterion(outputs.float(), labels)

        val_loss += loss.detach()
        num_batches += 1

        probs = torch.sigmoid(outputs)
        y_pred.append(probs.detach().cpu().numpy())
        y_true.append(labels.detach().cpu().numpy())

    y_pred = np.vstack(y_pred)
    y_true = np.vstack(y_true)

    per_label_auc = {}
    for i, col in enumerate(label_cols):
        y_va = y_true[:, i]
        if len(np.unique(y_va)) < 2:
            continue  # AUC undefined with only one class present
        per_label_auc[col] = roc_auc_score(y_va, y_pred[:, i])

    macro_auc = float(np.mean(list(per_label_auc.values()))) if per_label_auc else float("nan")
    avg_val_loss = (val_loss / num_batches).item()

    return avg_val_loss, macro_auc, per_label_auc




In [ ]:
# ==========================================
# Classifier training loop (TPU) -- checkpointing + early stopping,
# wrapped in MLflow
# ==========================================
# Fixes the "EPOCHS defined then ignored" bug (loop previously always ran
# a hardcoded range(100) regardless of the EPOCHS variable) and adds
# validation-AUROC-driven early stopping (there was none before -- the
# model that got evaluated downstream was simply whatever epoch 100
# happened to leave it at).

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
optimizer = torch.optim.Adam(
    [p for p in clf_model.parameters() if p.requires_grad], lr=CFG["clf_lr"]
)

CLF_CHECKPOINT_DIR = Path("checkpoints_classifier_simclr")
CLF_CHECKPOINT_DIR.mkdir(exist_ok=True)

EPOCHS = CFG["clf_epochs"]
PATIENCE = CFG["clf_patience"]

best_val_auc = -1.0
patience_counter = 0
train_history = []
val_history = []

with mlflow.start_run(run_name="simclr_classifier"):
    mlflow.log_params({
        "encoder": "SimCLR",
        "frozen_encoder": True,
        "batch_size": CFG["clf_batch_size"],
        "lr": CFG["clf_lr"],
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "n_classes": len(SUPERCLASSES),
    })

    for epoch in range(EPOCHS):
        train_loss = train_clf_epoch(clf_model, clf_train_loader, optimizer, criterion)
        val_loss, val_auc, per_label_auc = evaluate_clf(clf_model, clf_valid_loader, criterion)

        train_history.append(train_loss)
        val_history.append((val_loss, val_auc))
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss, "val_auc": val_auc}, step=epoch)

        xm.master_print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro AUC: {val_auc:.4f}"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            xm.save(clf_model.state_dict(), CLF_CHECKPOINT_DIR / "classifier_best.pth")
            xm.master_print(f"      New best classifier saved (Macro AUC={best_val_auc:.4f})")
        else:
            patience_counter += 1
            xm.master_print(f"      No improvement ({patience_counter}/{PATIENCE})")
            if patience_counter >= PATIENCE:
                xm.master_print(f"Early stopping at epoch {epoch + 1}")
                break

    mlflow.log_metric("best_val_auc", best_val_auc)
    mlflow.log_artifact(str(CLF_CHECKPOINT_DIR / "classifier_best.pth"))




In [ ]:
# ==========================================
# Final macro + per-label AUC (best checkpoint, not last epoch)
# ==========================================

clf_model.load_state_dict(torch.load(CLF_CHECKPOINT_DIR / "classifier_best.pth", map_location="cpu"))
clf_model = clf_model.to(DEVICE)

val_loss, val_auc, per_label_auc = evaluate_clf(clf_model, clf_valid_loader, criterion)

print(f"Val Loss  = {val_loss:.4f}")
print(f"Macro AUC = {val_auc:.4f}")
print()
print("Per-label AUC:")
for col, score in per_label_auc.items():
    print(f"  {col}: {score:.4f}")

skipped = [c for c in SUPERCLASSES if c not in per_label_auc]
if skipped:
    print(f"\n(skipped, insufficient class variance in this split: {skipped})")




In [ ]:
# ==========================================
# SupCon projection head (UNCHANGED architecture; renamed to match the
# image notebook's SupConProjectionHead naming convention)
# ==========================================

class SupConProjectionHead(nn.Module):
    def __init__(self, in_dim=CFG["embedding_dim"], proj_dim=CFG["projection_dim"]):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, proj_dim),
        )

    def forward(self, x):
        z = self.net(x)
        z = F.normalize(z, p=2, dim=1)  # L2-normalize here, once
        return z


class ECGSupConModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.projector = SupConProjectionHead(encoder.feature_dim, CFG["projection_dim"])

    def forward(self, x):
        h = self.encoder(x)   # (B, 256)
        z = self.projector(h) # (B, 128) -- already L2-normalized by the projector
        return h, z




In [ ]:
# ==========================================
# Multi-label SupCon loss (UNCHANGED math)
# ==========================================

def label_similarity(y):
    """y: (B, C) multi-hot -> returns: (B, B) Jaccard-style label similarity"""
    y = y.float()
    intersection = torch.matmul(y, y.T)
    y_sum = y.sum(dim=1, keepdim=True)
    union = y_sum + y_sum.T - intersection + 1e-8
    return intersection / union


def multilabel_supcon_loss(z, y, temperature=CFG["supcon_temperature"]):
    """z: (2B, D) L2-normalized embeddings (both views). y: (B, C) multi-hot
    labels for one view (duplicated to 2B)."""
    B = y.size(0)
    y = torch.cat([y, y], dim=0)  # (2B, C) -- same label for both views

    sim_labels = label_similarity(y)                # (2B, 2B)
    sim = torch.matmul(z, z.T) / temperature         # (2B, 2B)

    mask = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, torch.finfo(sim.dtype).min)
    sim_labels = sim_labels.masked_fill(mask, 0)

    row_sum = sim_labels.sum(dim=1, keepdim=True)
    sim_labels = sim_labels / (row_sum + 1e-6)
    valid = (row_sum > 1e-6).float()

    log_prob = F.log_softmax(sim, dim=1)
    loss = -(sim_labels * log_prob).sum(dim=1)
    return (loss * valid.squeeze()).sum() / (valid.sum() + 1e-6)




In [ ]:
# ==========================================
# MultiSupCon dataset + TPU dataloader (UNCHANGED augmentations/dataset)
# ==========================================
# The label-vector `lexsort` that used to run here (indices =
# np.lexsort(y_train_supacon.to_numpy().T); arrays reordered by it) had
# zero effect at runtime: the DataLoader below uses shuffle=True, so
# batches are drawn in random order every epoch regardless of the
# underlying array order. Removed as dead code -- no functional change.

class ECGSSLWithLabels(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def augment(self, x):
        x = x.clone()

        # noise
        x = x + 0.01 * torch.randn_like(x)

        # time shift
        shift = torch.randint(-150, 150, (1,))
        x = torch.roll(x, shifts=int(shift), dims=1)

        # scaling
        scale = torch.rand(1) * 0.4 + 0.8
        x = x * scale

        # one strong transform
        if torch.rand(1) < 0.5:
            length = torch.randint(150, 400, (1,))
            start = torch.randint(0, x.shape[1] - length, (1,))
            x[:, start:start + length] = 0
        else:
            lead = torch.randint(0, 12, (1,))
            x[lead] = 0

        return x

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.Y[idx]
        x1 = self.augment(x)
        x2 = self.augment(x)
        return x1, x2, y


y_train_supcon = y_train[SUPERCLASSES]
x_train_supcon = x_train

supcon_dataset = ECGSSLWithLabels(x_train_supcon, y_train_supcon.to_numpy())

supcon_DataLoader = DataLoader(
    supcon_dataset,
    batch_size=CFG["supcon_batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    persistent_workers=True,
    drop_last=True,
)
supcon_loader = pl.MpDeviceLoader(supcon_DataLoader, DEVICE)

x1, x2, labels = next(iter(supcon_loader))
print(f"View1  : {x1.shape}")
print(f"View2  : {x2.shape}")
print(f"Labels : {labels.shape} -> (batch, {len(SUPERCLASSES)} classes)")




In [ ]:
# ==========================================
# SupCon model + optimizer + scheduler
# ==========================================
# Fixed: the original scheduler was CosineAnnealingLR(T_max=20) but the
# loop ran for EPOCHS=100 -- the schedule fully decayed at epoch 20, then
# the remaining 80 epochs trained on a stale/cyclically-repeating LR that
# no longer matched the intended one-shot decay. T_max now spans the full
# training run, and a short warmup is added (same warmup+cosine pattern
# as the SimCLR phase / image notebook), matching item 7 of the spec.

encoder = ECGEncoder().to(DEVICE)
model = ECGSupConModel(encoder).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=CFG["supcon_lr"], weight_decay=CFG["supcon_weight_decay"]
)

warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, total_iters=CFG["supcon_warmup_epochs"]
)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["supcon_epochs"] - CFG["supcon_warmup_epochs"]
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup, cosine], milestones=[CFG["supcon_warmup_epochs"]]
)

x1, x2, labels = next(iter(supcon_loader))
images = torch.cat([x1, x2], dim=0)
with torch.no_grad():
    _, z = model(images)
print("Sanity loss:", multilabel_supcon_loss(z.float(), labels, temperature=CFG["supcon_temperature"]).item())




In [ ]:
# ==========================================
# MultiSupCon training step (TPU, bf16 autocast) -- same pattern as
# train_one_epoch, applied to the SupCon objective
# ==========================================

def train_supcon_epoch(model, loader, optimizer, temperature=CFG["supcon_temperature"], log_every=50):
    model.train()
    running_loss = torch.zeros((), device=DEVICE)
    num_batches = 0

    progress_bar = tqdm(loader)
    for x1, x2, labels_ in progress_bar:
        images = torch.cat([x1, x2], dim=0)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="xla", dtype=torch.bfloat16):
            _, z = model(images)
            loss = multilabel_supcon_loss(z.float(), labels_, temperature=temperature)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        xm.optimizer_step(optimizer, barrier=True)
        # (no separate xm.mark_step() -- barrier=True already does that)

        running_loss += loss.detach()
        num_batches += 1

        if num_batches % log_every == 0:
            progress_bar.set_postfix(loss=f"{(running_loss / num_batches).item():.4f}")

    return (running_loss / num_batches).item()




In [ ]:
# ==========================================
# Checkpoint config + resume (defined BEFORE the training loop, same
# fixed ordering as Section G)
# ==========================================

CHECKPOINT_DIR = Path("checkpoints_supcon")
CHECKPOINT_DIR.mkdir(exist_ok=True)

CHECKPOINT_EVERY = CFG["checkpoint_every"]
EVAL_EVERY = CFG["eval_every"]
PATIENCE = CFG["pretrain_patience"]

START_EPOCH = 0
history = []
probe_history = []
best_auroc = -1.0
patience_counter = 0

resume_path = CHECKPOINT_DIR / "latest.pth"
if resume_path.exists():
    ckpt = torch.load(resume_path, map_location="cpu")

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])

    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    scheduler.load_state_dict(ckpt["scheduler_state"])
    history = ckpt["history"]
    probe_history = ckpt.get("probe_history", [])
    best_auroc = ckpt.get("best_auroc", -1.0)
    patience_counter = ckpt.get("patience_counter", 0)
    START_EPOCH = ckpt["epoch"]

    xm.master_print(f"Resumed from checkpoint at epoch {START_EPOCH}")
else:
    xm.master_print("No checkpoint found -- starting fresh")




In [ ]:
# ==========================================
# MultiSupCon training loop (TPU) -- resume + checkpointing + AUROC-based
# early stopping, wrapped in a single MLflow run
# ==========================================
# NOTE (research decision, intentionally left unchanged): the encoder
# here is re-initialized from scratch (`ECGEncoder().to(DEVICE)` in the
# previous cell), not warm-started from the SimCLR checkpoint, exactly as
# in the original notebook. Whether MultiSupCon should fine-tune the
# SimCLR-pretrained encoder or train an independent one is a research
# design decision (see report), not an engineering one -- left as-is per
# the "do not change methodology" instruction.

with mlflow.start_run(run_name="multisupcon_pretrain"):
    mlflow.log_params({
        "encoder": "ECGEncoder",
        "ssl_method": "MultiSupCon",
        "warm_started_from_simclr": False,
        "batch_size": CFG["supcon_batch_size"],
        "lr": CFG["supcon_lr"],
        "weight_decay": CFG["supcon_weight_decay"],
        "temperature": CFG["supcon_temperature"],
        "epochs": CFG["supcon_epochs"],
        "warmup_epochs": CFG["supcon_warmup_epochs"],
        "eval_every": EVAL_EVERY,
        "patience": PATIENCE,
    })

    for epoch in range(START_EPOCH, CFG["supcon_epochs"]):
        train_loss = train_supcon_epoch(model, supcon_loader, optimizer, temperature=CFG["supcon_temperature"])
        scheduler.step()
        history.append(train_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        xm.master_print(
            f"Epoch [{epoch + 1}/{CFG['supcon_epochs']}] "
            f"Loss: {train_loss:.4f} | LR: {current_lr:.2e}"
        )
        mlflow.log_metrics({"supcon_loss": train_loss, "lr": current_lr}, step=epoch)

        if (epoch + 1) % EVAL_EVERY == 0 or (epoch + 1) == CFG["supcon_epochs"]:
            auroc = linear_probe_auroc(model.encoder)
            probe_history.append((epoch + 1, auroc))
            mlflow.log_metric("probe_auroc", auroc, step=epoch)
            xm.master_print(f"      Probe AUROC: {auroc:.4f}")

            if auroc > best_auroc:
                best_auroc = auroc
                patience_counter = 0
                xm.save(model.encoder.state_dict(), CHECKPOINT_DIR / "encoder_best.pth")
                xm.master_print(f"      New best encoder saved (AUROC={best_auroc:.4f})")
            else:
                patience_counter += 1
                xm.master_print(f"      No improvement ({patience_counter}/{PATIENCE})")
                if patience_counter >= PATIENCE:
                    xm.master_print(f"Early stopping at epoch {epoch + 1}")
                    xm.save({
                        "epoch": epoch + 1,
                        "model_state": model.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "scheduler_state": scheduler.state_dict(),
                        "history": history,
                        "probe_history": probe_history,
                        "best_auroc": best_auroc,
                        "patience_counter": patience_counter,
                    }, CHECKPOINT_DIR / "latest.pth")
                    break

        if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == CFG["supcon_epochs"]:
            xm.save({
                "epoch": epoch + 1,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "history": history,
                "probe_history": probe_history,
                "best_auroc": best_auroc,
                "patience_counter": patience_counter,
            }, CHECKPOINT_DIR / "latest.pth")

    mlflow.log_metric("best_probe_auroc", best_auroc)
    mlflow.log_artifact(str(CHECKPOINT_DIR / "encoder_best.pth"))

# Debug-only snapshot of the final epoch's weights -- NOT used downstream.
xm.save(model.encoder.state_dict(), "ecg_encoder_multisupcon_final.pth")




In [ ]:
# ==========================================
# Load frozen MultiSupCon encoder -- BEST checkpoint, not last epoch
# ==========================================

encoder = ECGEncoder()
encoder.load_state_dict(torch.load(CHECKPOINT_DIR / "encoder_best.pth", map_location="cpu"))
encoder = encoder.to(DEVICE)

for p in encoder.parameters():
    p.requires_grad = False
encoder.eval()

clf_model = ECGClassifier(encoder).to(DEVICE)

total_params = sum(p.numel() for p in clf_model.parameters())
trainable_params = sum(p.numel() for p in clf_model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")




In [ ]:
# ==========================================
# Classifier training loop (TPU) -- checkpointing + early stopping,
# wrapped in MLflow. Same helpers as the SimCLR-classifier phase above.
# ==========================================

criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
optimizer = torch.optim.Adam(
    [p for p in clf_model.parameters() if p.requires_grad], lr=CFG["clf_lr"]
)

CLF_CHECKPOINT_DIR = Path("checkpoints_classifier_supcon")
CLF_CHECKPOINT_DIR.mkdir(exist_ok=True)

EPOCHS = CFG["clf_epochs"]
PATIENCE = CFG["clf_patience"]

best_val_auc = -1.0
patience_counter = 0
train_history = []
val_history = []

with mlflow.start_run(run_name="multisupcon_classifier"):
    mlflow.log_params({
        "encoder": "MultiSupCon",
        "frozen_encoder": True,
        "batch_size": CFG["clf_batch_size"],
        "lr": CFG["clf_lr"],
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "n_classes": len(SUPERCLASSES),
    })

    for epoch in range(EPOCHS):
        train_loss = train_clf_epoch(clf_model, clf_train_loader, optimizer, criterion)
        val_loss, val_auc, per_label_auc = evaluate_clf(clf_model, clf_valid_loader, criterion)

        train_history.append(train_loss)
        val_history.append((val_loss, val_auc))
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss, "val_auc": val_auc}, step=epoch)

        xm.master_print(
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro AUC: {val_auc:.4f}"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
            xm.save(clf_model.state_dict(), CLF_CHECKPOINT_DIR / "classifier_best.pth")
            xm.master_print(f"      New best classifier saved (Macro AUC={best_val_auc:.4f})")
        else:
            patience_counter += 1
            xm.master_print(f"      No improvement ({patience_counter}/{PATIENCE})")
            if patience_counter >= PATIENCE:
                xm.master_print(f"Early stopping at epoch {epoch + 1}")
                break

    mlflow.log_metric("best_val_auc", best_val_auc)
    mlflow.log_artifact(str(CLF_CHECKPOINT_DIR / "classifier_best.pth"))




In [ ]:
# ==========================================
# Final macro + per-label AUC (best checkpoint, not last epoch)
# ==========================================

clf_model.load_state_dict(torch.load(CLF_CHECKPOINT_DIR / "classifier_best.pth", map_location="cpu"))
clf_model = clf_model.to(DEVICE)

val_loss, val_auc, per_label_auc = evaluate_clf(clf_model, clf_valid_loader, criterion)

print(f"Val Loss  = {val_loss:.4f}")
print(f"Macro AUC = {val_auc:.4f}")
print()
print("Per-label AUC:")
for col, score in per_label_auc.items():
    print(f"  {col}: {score:.4f}")

skipped = [c for c in SUPERCLASSES if c not in per_label_auc]
if skipped:
    print(f"\n(skipped, insufficient class variance in this split: {skipped})")




In [ ]:
# ==========================================
# Consolidate final artifacts
# ==========================================

import shutil

SAVE_DIR = Path("/kaggle/working/final_models")
SAVE_DIR.mkdir(exist_ok=True, parents=True)

shutil.copy("checkpoints_simclr/encoder_best.pth", SAVE_DIR / "encoder_simclr_best.pth")
shutil.copy("checkpoints_supcon/encoder_best.pth", SAVE_DIR / "encoder_multisupcon_best.pth")
shutil.copy("checkpoints_classifier_simclr/classifier_best.pth", SAVE_DIR / "classifier_simclr_best.pth")
shutil.copy("checkpoints_classifier_supcon/classifier_best.pth", SAVE_DIR / "classifier_multisupcon_best.pth")

print("Saved to:", SAVE_DIR)
print(os.listdir(SAVE_DIR))